# 20. 검증셋 샘플링 및 통합 저장 (Sampled Validation Set)

이 노트북은 Optuna 하이퍼파라미터 튜닝의 연산 효율성을 위해 **검증 데이터셋을 전략적으로 샘플링**하여 저장합니다.

**핵심 전략:**
1. **Sampled Validation**: 검증셋의 모든 고장 개체 중 **80%**만 추출하고, 정상 개체는 추출된 고장 개체 수의 **10배수**만큼 랜덤 샘플링하여 병합합니다.
2. **개체 단위 샘플링**: 모든 샘플링은 개체(serial_number) 단위로 이루어지며, 시계열 타임라인이 끊기지 않고 완전하게 유지됩니다. 기존의 near-failure 가중치 등은 제외합니다.
3. **Optuna 내부 루프 전용**: 이 데이터셋을 활용해 Optuna 내부 튜닝 루프를 빠르게 수행하며, 상위 후보군에 대한 최종 리랭킹 시에만 전체 검증셋을 사용합니다.

In [ ]:
import sys, os
import pandas as pd
import numpy as np
from pathlib import Path

# ── [독립 설정 구역] ────────────────────────────────────────
# 학습 컨피그와 완전히 독립적으로 실행되도록 경로와 설정을 로컬에 직접 지정합니다.

DATA_ROOT     = "../data2"      # 데이터 루트 폴더 (data 또는 data2)
VAL_TUNE_PATH = f"{DATA_ROOT}/03_splitting/val_tune.parquet"

TARGET_SEED = 42      # 샘플링 시드 번호
VAL_FAILED_RATIO = 0.8  # 고장 개체 추출 비율 (80%)
VAL_NEG_RATIO = 3        # 정상:고장 개체수 비율 (3배수)

# [저장 정보]
# 학습용 서브셋과 동일한 폴더에 'val_sampled.parquet' 이름으로 저장됩니다.
SAVE_DIR = Path(DATA_ROOT) / "06_subset_generation" / f"seed_{TARGET_SEED}"
SAVE_PATH = SAVE_DIR / "val_sampled.parquet"

print(f"🚀 검증셋 샘플링 준비: Seed={TARGET_SEED}")
print(f"📂 통합 저장 위치: {SAVE_PATH.resolve()}")

## 1. 원본 검증 데이터 로드

In [ ]:
print(f"📂 원본 데이터 로딩 중... ({VAL_TUNE_PATH})")
df_val = pd.read_parquet(VAL_TUNE_PATH)
print(f"✅ 로드 완료: {len(df_val):,} rows")

## 2. 개체 단위 1:10 검증 데이터셋 샘플링 실행

In [ ]:
print("🔧 검증 데이터셋 개체 단위 샘플링 중 (고장 개체 80%, 정상 개체 10배수)...\n")

# 1. 고장 개체와 정상 개체의 serial_number 목록 추출
# 고장이 단 한 번이라도 발생한 개체는 고장 개체, 그렇지 않으면 정상 개체로 분류
failed_serials = df_val[df_val['failure'] == 1]['serial_number'].unique()
all_serials = df_val['serial_number'].unique()
healthy_serials = np.array(list(set(all_serials) - set(failed_serials)))

n_failed_orig = len(failed_serials)
n_healthy_orig = len(healthy_serials)

print(f"  * 원본 검증셋 유니크 개체 수: {len(all_serials):,}개")
print(f"  * 원본 검증셋 고장 개체 수: {n_failed_orig:,}개")
print(f"  * 원본 검증셋 정상 개체 수: {n_healthy_orig:,}개")

# 2. 고장 개체 중 80% 추출 (seed=TARGET_SEED)
n_failed_select = int(n_failed_orig * VAL_FAILED_RATIO)
rng = np.random.default_rng(TARGET_SEED)
sampled_failed = rng.choice(failed_serials, size=n_failed_select, replace=False)

# 3. 추출된 고장 개체의 3배수 만큼 정상 개체 추출 (seed=TARGET_SEED)
n_healthy_select = n_failed_select * VAL_NEG_RATIO
assert n_healthy_orig >= n_healthy_select, f"오류: 원본 정상 개체 수({n_healthy_orig})가 필요한 수({n_healthy_select})보다 적습니다."
sampled_healthy = rng.choice(healthy_serials, size=n_healthy_select, replace=False)

print(f"  * 샘플링할 고장 개체 수: {n_failed_select:,}개 (80%)")
print(f"  * 샘플링할 정상 개체 수: {n_healthy_select:,}개 (고장의 3배수)")

# 4. 고장 및 정상 개체 병합
selected_serials = set(sampled_failed).union(set(sampled_healthy))

# 5. 전체 데이터에서 해당 개체들의 시계열 로우 추출 및 셔플
df_sampled_val = df_val[df_val['serial_number'].isin(selected_serials)].copy()
df_sampled_val = df_sampled_val.sample(frac=1, random_state=TARGET_SEED).reset_index(drop=True)

print(f"\n✅ 샘플링 완료: {len(df_sampled_val):,} rows")

## 3. 학습 세트 폴더에 통합 저장

In [ ]:
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 학습용 데이터(subset_*.parquet)와 함께 관리됩니다.
df_sampled_val.to_parquet(SAVE_PATH, index=False)

print(f"✨ 샘플링된 검증셋 저장 완료!")
print(f"📍 경로: {SAVE_PATH.resolve()}")
print(f"🔗 이 데이터는 이제 '{SAVE_DIR.name}' 세트의 일부로 학습 시 사용됩니다.")

## 4. 샘플링 검증셋 결과 및 개체 비율 무결성 검증 테스트 (Verification Tests)

생성된 `val_sampled.parquet` 검증셋의 실제 물리 파일 존재 여부, 고장 개체의 80% 샘플링 정합성, 정상 개체의 10배수 비율 정합성, 그리고 정상 개체 데이터 내에 고장 기록(failure=1)이 존재하지 않는지를 수학적으로 엄밀히 입증합니다.

In [ ]:
print("🔍 [6-C단계 무결성 검증] 시작...")

try:
    # 1. 파일 물리 존재 여부
    print("Test 1: 샘플링 검증셋 물리 파일 존재 검증")
    assert SAVE_PATH.is_file(), f"오류: {SAVE_PATH} 파일이 생성되지 않았습니다."
    print("  -> [PASS] 샘플링 검증셋 파일 존재 확인.")

    # 2. 고장 개체 및 정상 개체 샘플링 수 및 1:3 비율 검증
    print("Test 2: 고장 개체 80% 추출 및 정상 개체 3배수 비율 검증")
    df_check = pd.read_parquet(SAVE_PATH)
    
    check_failed = df_check[df_check['failure'] == 1]['serial_number'].unique()
    check_healthy = np.array(list(set(df_check['serial_number'].unique()) - set(check_failed)))
    
    # 원본 고장 개체 수 로드
    df_val_orig = pd.read_parquet(VAL_TUNE_PATH)
    orig_failed_cnt = len(df_val_orig[df_val_orig['failure'] == 1]['serial_number'].unique())
    
    expected_failed_cnt = int(orig_failed_cnt * VAL_FAILED_RATIO)
    expected_healthy_cnt = expected_failed_cnt * VAL_NEG_RATIO
    
    assert len(check_failed) == expected_failed_cnt, f"오류: 고장 개체 수({len(check_failed)})가 예상치({expected_failed_cnt})와 다릅니다!"
    assert len(check_healthy) == expected_healthy_cnt, f"오류: 정상 개체 수({len(check_healthy)})가 예상치({expected_healthy_cnt})와 다릅니다!"
    print(f"  -> [PASS] 고장 개체 {len(check_failed)}개 (80%), 정상 개체 {len(check_healthy)}개 (3배수) 비율 완벽 일치.")

    # 3. 정상 개체의 순수성 검증
    print("Test 3: 정상 개체 타임라인 내 고장 기록(failure=1) 부존재 검증")
    df_healthy_rows = df_check[~df_check['serial_number'].isin(check_failed)]
    assert (df_healthy_rows['failure'] == 0).all(), "오류: 정상 개체 데이터 중 failure=1인 행이 존재합니다!"
    print("  -> [PASS] 정상 개체의 고장 기록(failure=1) 부존재 확인.")

    print("\n🏆 [6-C단계 정합성 검증 완료] 모든 엄격한 테스트 조건을 만족합니다!")
finally:
    pass